# Logistic regression: scores, sigmoid, and decisions

Companion to HTB Academy Module 290, Section 5. This is a small **hand-weighted demonstration**, not a trained spam filter. No real email, target, or challenge data is used.

Read $z=m_1x_1+m_2x_2+c$ as *z equals m sub one times x sub one plus m sub two times x sub two plus c*. The $x$ values are features, the $m$ values are weights, and $c$ is an intercept. $z$ is a raw score, not a probability.

In [ ]:
import numpy as np

# Each row is one imaginary email: [number of links, suspicious-word indicator].
# The second column is 0 or 1; these are only toy features.
features = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [3.0, 1.0]])
weights = np.array([1.2, 0.8])
intercept = -2.0
assert features.shape == (4, 2)  # Four examples, two features each.
assert weights.shape == (2,)

# Formula: z_i = m_1*x_i1 + m_2*x_i2 + c.
# The matrix product computes both weighted contributions for each row.
scores = features @ weights + intercept
print('raw scores:', scores)
assert np.allclose(scores, [-2.0, -0.8, 0.0, 2.4])

## Convert scores to probability-shaped values

Read $p(x)=\sigma(z)=1/(1+e^{-z})$ as *p of x equals sigma of z equals one divided by one plus e to the negative z*. The sigmoid maps any raw score into the open interval `(0, 1)`. The numbers below are **model scores**, not validated real-world spam probabilities.

In [ ]:
# Formula: p_i = 1 / (1 + exp(-z_i)). Toy scores are small, so this simple form is safe here.
probabilities = 1.0 / (1.0 + np.exp(-scores))
print('class-1 scores:', np.round(probabilities, 3))
assert np.all((probabilities > 0.0) & (probabilities < 1.0))
assert np.isclose(probabilities[2], 0.5)  # z=0 maps to the midpoint.

# Odds are p/(1-p); taking their natural log recovers z for a sigmoid score.
log_odds = np.log(probabilities / (1.0 - probabilities))
assert np.allclose(log_odds, scores)

## Apply a decision threshold

Read $\hat y=1$ when $p(x)\ge t$ as *y-hat is one when p of x is greater than or equal to threshold t*. Raising $t$ changes the decision policy, not the weights or scores. The first threshold below includes a score exactly equal to `0.5`; the `>=` tie rule is deliberate.

In [ ]:
for threshold in (0.5, 0.8):
    predictions = (probabilities >= threshold).astype(int)
    # Formula: boundary score z = log(t / (1-t)), where t is this threshold.
    boundary_score = np.log(threshold / (1.0 - threshold))
    print(f'threshold={threshold}: labels={predictions}, boundary z={boundary_score:.3f}')
    if threshold == 0.5:
        assert predictions.tolist() == [0, 0, 1, 1]
        assert np.isclose(boundary_score, 0.0)
    else:
        assert predictions.tolist() == [0, 0, 0, 1]
        assert np.isclose(boundary_score, np.log(4.0))

The threshold `0.8` requires a higher raw score than `0.5`. In a two-feature plot, a fixed raw-score equation such as $1.2x_1+0.8x_2-2=0$ is a line; in higher dimensions it is a flat hyperplane. These examples have no ground-truth labels, so we cannot calculate precision, recall, or whether either threshold is operationally better.